<p style="text-align: center">
<img src="../../assets/images/dtlogo.png" alt="Duckietown" width="50%">
</p>

# Guida Completa ed Intuitiva al Filtro a Particelle (Particle Filter)
### Spiegazione passo-passo "da zero" per la localizzazione non gaussiana di un robot mobile (Duckiebot)

Benvenuto nel secondo notebook della serie sulla localizzazione probabilistica! Nel notebook precedente abbiamo studiato il **Filtro di Kalman (KF)**. Il Filtro di Kalman è uno strumento matematico potentissimo, ma ha un grande "tallone d'Achille":
* Funziona **solo** se il mondo è **lineare** e se l'incertezza può essere rappresentata da **una singola campana gaussiana (unimodale)**.

### Il Limite del Filtro di Kalman nel mondo reale:
Immagina che il tuo Duckiebot entri in una stanza rettangolare simmetrica che ha due porte identiche. Il sensore legge: *"Vedo una porta a 1 metro di distanza"*.
* Dov'è il robot? Potrebbe essere davanti alla **Porta A**, oppure davanti alla **Porta B**!
* La probabilità reale ha **due picchi** (distribuzione bimodal/multimodale).
* Il Filtro di Kalman non può gestire due picchi: è costretto a calcolare la media tra le due porte, posizionando la sua stima... **esattamente al centro del muro, dove non c'è nessuna porta!**

### La Soluzione Rivoluzionaria: Il Filtro a Particelle (Metodo Monte Carlo)
Invece di descrivere la posizione con una formula matematica rigida (media e covarianza), il **Filtro a Particelle (Particle Filter - PF)** usa una strategia geniale e intuitiva:
> **Rilasciamo nell'ambiente un "esercito" di $M$ robot virtuali (chiamati particelle o ipotesi).**

Ogni particella $i$ possiede:
1. Uno **stato ipotizzato** $\mathbf{x}_t^i$ (es. una coordinata $(x, y)$ sul pavimento).
2. Un **peso di importanza** $w_t^i$ (un punteggio che dice: *"Quanto è credibile questa particella alla luce delle misure reali?"*).

In questa guida smonteremo il Particle Filter passo dopo passo con esempi concreti sul Duckiebot, formule spiegate a parole semplici e codice Python funzionante e commentato riga per riga.

In [ ]:
%matplotlib inline
import time
import random
import copy
import numpy as np
import scipy.stats
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from IPython import display
from tqdm.contrib import tzip

# Fissiamo i semi di generatore casuale per garantire la riproducibilita didattica
np.random.seed(42)
random.seed(42)
print("Tutte le librerie sono state importate con successo!")

---
# I 4 Passi Fondamentali dell'Algoritmo del Particle Filter

Il ciclo di vita di un Filtro a Particelle si ripete ad ogni istante di tempo $t$ attraverso 4 fasi sequenziali:

```
  [1. Inizializzazione] ---> [2. Predizione / Moto]
                                    |
                                    v
  [4. Ricampionamento]  <--- [3. Misura / Pesi]
```

1. **Inizializzazione (Sampling Iniziale):**
   * All'inizio non sappiamo dove si trovi il Duckiebot sulla pista.
   * Generiamo $M$ particelle distribuite in modo **uniforme e casuale** su tutta la mappa.
   * A ciascuna particella assegniamo inizialmente lo stesso peso: $w_0^i = 1$ (o $w_0^i = 1/M$).

2. **Predizione (Prediction Step / Motion Update):**
   * Inviamo al Duckiebot un comando di controllo $\mathbf{u}_t$ (es. velocità $v_x, v_y$ per 1 secondo).
   * Spostiamo **ciascuna particella** facendola avanzare secondo la fisica del moto, aggiungendo a ciascuna un pizzico di **rumore casuale** per simulare lo slittamento delle ruote:
     $$\mathbf{x}_t^i = \mathbf{x}_{t-1}^i + \mathbf{u}_t + \mathbf{w}_t^i, \quad \mathbf{w}_t^i \sim \mathcal{N}(0, \sigma_R^2 I)$$
   * Le particelle si muovono e la "nuvola" si allarga leggermente.

3. **Aggiornamento dei Pesi (Measurement Update):**
   * Il robot reale acquisisce una misura con i suoi sensori: $\mathbf{z}_t$.
   * Ora interroghiamo ogni singola particella: *"Se il robot fossi tu, quanto sarebbe probabile leggere proprio la misura $\mathbf{z}_t$?"*.
   * Questa probabilità è la **verosimiglianza** (likelihood) $p(\mathbf{z}_t | \mathbf{x}_t^i)$.
   * Il nuovo peso della particella diventa proporzionale a questo valore:
     $$w_t^i = p(\mathbf{z}_t | \mathbf{x}_t^i)$$
   * Le particelle vicine alla posizione reale del robot ottengono pesi altissimi! Le particelle lontane ricevono pesi quasi pari a zero.

4. **Ricampionamento (Resampling / Roulette Wheel):**
   * Eliminiamo le particelle che hanno "indovinato male" (peso basso) e moltiplichiamo quelle che hanno "indovinato bene" (peso alto).
   * Estraiamo $M$ nuove particelle pescando dal vecchio insieme con probabilità proporzionale ai pesi $w_t^i$.
   * La nuvola di particelle collassa rapidamente attorno alla vera posizione del robot!

5. **Stima Definitiva dello Stato:**
   * La stima della posizione del robot si ottiene calcolando la **media pesata** (o la media delle particelle ricampionate):
     $$\hat{\mathbf{x}}_t = \frac{1}{M} \sum_{i=1}^M \mathbf{x}_t^i$$

---
## La Sfida del *Particle Depletion* e la Soluzione $\alpha$

C'è un potenziale problema nel ricampionamento puro:
* Se all'inizio nessuna particella è stata generata vicino alla vera posizione del robot, oppure se il robot viene sollevato e spostato di colpo da un essere umano (problema del *Kidnapped Robot* / robot rapito), tutte le particelle esistenti avranno un peso bassissimo.
* Dopo pochi passi di ricampionamento, le particelle si concentrano tutte in un unico punto errato e **muoiono senza potersi più riprendere** (*Particle Depletion* o impoverimento delle particelle).

### Il Trucco dell'Iniezione Casuale (Parametro $\alpha$):
Per risolvere brillantemente questo problema:
* Ricampioniamo una percentuale maggioritaria (ad esempio $\alpha = 95\% = 0.95$) di particelle basandoci sui pesi delle particelle attuali.
* Generiamo il restante $(1 - \alpha) = 5\%$ di particelle **completamente da capo in modo uniforme e casuale nella mappa**.
* Questo garantisce al robot una capacità perenne di "esplorare" e di riprendersi anche se si fosse completamente perso!

---
# Esempio Pratico: Localizzazione 2D del Duckiebot

Consideriamo il problema proposto nel notebook originale:
* Un robot si muove in una stanza priva di ostacoli partendo dall'origine $(x, y) = (0, 0)$.
* Il robot applica comandi di velocità costante $v_x = v_y = 0.1\text{ m/s}$ lungo la diagonale per $N = 100$ passi temporali.
* A ciascun passo temporale il robot misura la propria posizione con sensori di distanza dalle pareti.

### Modello di Movimento:
$$\mathbf{x}_t = \mathbf{x}_{t-1} + \mathbf{u}_t + \mathbf{w}_t, \quad \mathbf{w}_t \sim \mathcal{N}([0, 0]^T, \sigma_R I_2)$$
con deviazione standard del rumore di movimento $\sigma_R = 0.01$.

### Modello di Misura:
$$\mathbf{z}_t = \mathbf{x}_t + \mathbf{n}_t, \quad \mathbf{n}_t \sim \mathcal{N}([0, 0]^T, \sigma_Q I_2)$$
con deviazione standard del rumore del sensore $\sigma_Q = 0.5$.

> **Nota fondamentale:**
> A differenza del Filtro di Kalman, nel Particle Filter **non c'è alcun obbligo matematico che i modelli siano lineari o che i rumori siano gaussiani**! Qui usiamo modelli semplici per rendere immediato il confronto didattico, ma il Particle Filter può gestire qualsiasi funzione complessa o trigonometrica.

In [ ]:
# Definizione parametri temporali e traiettoria reale ideale
num_data = 100  # Numero di passi temporali simulati
ground_truth_x = np.linspace(0, 10, num=num_data + 1)
ground_truth_y = ground_truth_x.copy()  # Movimento a 45 gradi: x = y

# Simulazione delle misure rumorose del sensore (Ground Truth + Rumore Gaussiano)
measurement_noise_x_var = 0.5  # Deviazione standard del sensore lungo x
measurement_noise_y_var = 0.5  # Deviazione standard del sensore lungo y
noise_x = np.random.normal(loc=0.0, scale=measurement_noise_x_var, size=num_data - 1)
noise_y = np.random.normal(loc=0.0, scale=measurement_noise_y_var, size=num_data - 1)
measurement_x = np.linspace(10 / num_data, 10, num=num_data - 1) + noise_x
measurement_y = np.linspace(10 / num_data, 10, num=num_data - 1) + noise_y

# Visualizziamo il problema: Traiettoria Reale vs Misure Grezze
plt.figure(figsize=(8, 8))
plt.plot(ground_truth_x, ground_truth_y, 'k--', linewidth=2, label='Traiettoria reale (Ground Truth)')
plt.plot(measurement_x, measurement_y, 'r.', alpha=0.6, label='Misure rumorose del sensore')
plt.title('Il Problema: Sensore molto rumoroso (sigma = 0.5)', fontsize=14)
plt.xlabel('Posizione X [m]', fontsize=12)
plt.ylabel('Posizione Y [m]', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6)
plt.gca().set_aspect('equal', adjustable='box')
plt.show()

Come puoi notare dal grafico, le misure del sensore sono molto disperse attorno alla diagonale ideale.
Ora costruiremo l'architettura del **Particle Filter** per ripulire questo rumore e stimare la posizione reale del robot con grande accuratezza!

---
# Implementazione della Classe `Particle` e del Ricampionamento

Dobbiamo definire due elementi chiave:
1. La classe **`Particle`**:
   * Ogni particella ha le coordinate di stato `self.state = np.array([x, y])` e un peso `self.weight`.
   * Il metodo **`predict(u_t)`**: sposta la particella di $\mathbf{u}_t$ aggiungendo rumore gaussiano con deviazione standard $\sigma_R = 0.01$.
   * Il metodo **`update(z_t)`**: calcola la verosimiglianza della misura reale $\mathbf{z}_t$ data la posizione della particella, usando la funzione di densità di probabilità gaussiana multivariata:
     `scipy.stats.multivariate_normal.pdf(self.state, mean=z_t, cov=Q)`
2. La funzione **`sample_particle(particles)`**:
   * Esegue l'estrazione stocastica (*Roulette Wheel Selection*): la probabilità di pescare una particella è proporzionale al suo peso normalizzato.

In [ ]:
# Parametri di rumore del modello di moto e del modello di misura
sigma_R = 0.01  # Deviazione standard rumore di movimento ruote
sigma_Q = 0.50  # Deviazione standard rumore dei sensori

In [ ]:
class Particle:
    def __init__(self, x, y, w=1.0):
        self.state = np.array([float(x), float(y)])
        self.weight = float(w)

    def predict(self, u_t):
        """
        Fase di Predizione: muove la particella secondo il comando u_t,
        aggiungendo rumore gaussiano per simulare l'imprecisione reale delle ruote.
        """
        motion_noise = np.random.normal(loc=0.0, scale=sigma_R, size=self.state.shape)
        self.state = self.state + u_t + motion_noise

    def update(self, z_t):
        """
        Fase di Aggiornamento: assegna alla particella un peso proporzionale alla
        probabilita di aver rilevato la misura z_t trovandosi nello stato self.state.
        """
        # Matrice di covarianza del sensore: Q = sigma_Q * I_2
        Q = sigma_Q * np.identity(2)
        self.weight = scipy.stats.multivariate_normal.pdf(self.state, mean=z_t, cov=Q)


def sample_particle(particles):
    """
    Estrae una particella con probabilita proporzionale al suo peso (Roulette Wheel).
    """
    total_weight = sum(p.weight for p in particles)
    if total_weight == 0:
        # Se tutti i pesi sono nulli (caso estremo), estrae uniformemente
        return random.choice(particles)
        
    r = random.uniform(0.0, total_weight)
    cumulative_weight = 0.0
    for p in particles:
        cumulative_weight += p.weight
        if cumulative_weight >= r:
            return p
            
    return particles[-1]

---
# Esecuzione del Filtro a Particelle nel Tempo

Ora assembliamo l'intero ciclo del Particle Filter per i $100$ passi temporali:
* **$M = 100$ particelle.**
* **$\alpha = 0.95$**: ad ogni passo il $95\%$ delle particelle viene ricampionato in base ai pesi, mentre il $5\%$ viene rigenerato casualmente per evitare il *depletion*.
* Ad ogni istante:
  1. Le particelle avanzano con `predict(u_t)`.
  2. Acquisiamo la misura $\mathbf{z}_t$ dal sensore.
  3. Aggiorniamo i pesi con `update(z_t)`.
  4. Creiamo il nuovo insieme di particelle ricampionate (`sample_particle`).
  5. Calcoliamo la stima finale media $(\hat{x}_t, \hat{y}_t)$.

In [ ]:
num_particles = 100  # Numero di particelle (ipotesi)
alpha = 0.95         # Quota di particelle ricampionate dai pesi (1-alpha casuali)

filtered_xs = []
filtered_ys = []
measurements = []
particles_over_time = []

# Funzione per generare una particella casuale uniforme nell'area di interesse (0..10)
def initialize_particle():
    x = np.random.uniform(0.0, 10.0)
    y = np.random.uniform(0.0, 10.0)
    w = 1.0
    return Particle(x, y, w)

# 1. Inizializziamo le particelle uniformemente
particles = [initialize_particle() for _ in range(num_particles)]

# Comando di controllo costante (spostamento per passo)
u_t = np.array([10.0 / num_data, 10.0 / num_data])

# Esecuzione del ciclo del Particle Filter
for i in range(num_data):
    # A. PREDICT: muoviamo tutte le particelle col modello di moto
    for p in particles:
        p.predict(u_t)

    # B. MISURA: simuliamo la lettura hardware del sensore
    measurement_noise_x = np.random.normal(loc=0.0, scale=measurement_noise_x_var)
    measurement_noise_y = np.random.normal(loc=0.0, scale=measurement_noise_y_var)
    measurement_x_new = ground_truth_x[i + 1] + measurement_noise_x
    measurement_y_new = ground_truth_y[i + 1] + measurement_noise_y
    z_t = np.array([measurement_x_new, measurement_y_new])
    measurements.append([measurement_x_new, measurement_y_new])

    # C. UPDATE: aggiorniamo i pesi delle particelle con la verosimiglianza di z_t
    for p in particles:
        p.update(z_t)

    # Salviamo le particelle per la visualizzazione
    particles_over_time.append(copy.deepcopy(particles))

    # D. RICAMPIONAMENTO:
    new_particles = []
    # alpha * M particelle ricampionate dai pesi migliori
    num_resampled = int(num_particles * alpha)
    for _ in range(num_resampled):
        p_sampled = copy.deepcopy(sample_particle(particles))
        p_sampled.weight = 1.0  # Reset del peso dopo il ricampionamento
        new_particles.append(p_sampled)
        
    # (1 - alpha) * M particelle nuove iniettate casualmente (anti-depletion)
    num_random = int(num_particles * (1.0 - alpha))
    for _ in range(num_random):
        new_particles.append(initialize_particle())

    particles = new_particles

    # E. CALCOLO DELLA STIMA MEDIA DELLO STATO (escludendo le iniezioni puramente casuali)
    xs = [p.state[0] for p in particles[:num_resampled]]
    ys = [p.state[1] for p in particles[:num_resampled]]
    estimated_x = np.mean(xs)
    estimated_y = np.mean(ys)

    filtered_xs.append(estimated_x)
    filtered_ys.append(estimated_y)

measurements = np.array(measurements)
filtered_states = np.column_stack((filtered_xs, filtered_ys))
ground_truth = np.column_stack((ground_truth_x, ground_truth_y))
print("Simulazione del Particle Filter completata con successo!")

---
# Risultati: Confronto Traiettoria Reale vs Misure vs Particle Filter

Visualizziamo l'intero percorso stimato dal Particle Filter a confronto con le misure grezze e la verità assoluta:

In [ ]:
plt.figure(figsize=(9, 9))
plt.plot(ground_truth[:, 0], ground_truth[:, 1], 'k--', linewidth=2.5, label='Traiettoria reale (Ground Truth)')
plt.plot(measurements[:, 0], measurements[:, 1], 'r.', alpha=0.5, label='Misure rumorose del sensore')
plt.plot(filtered_states[:, 0], filtered_states[:, 1], 'g-', linewidth=2.5, label='Stima del Particle Filter')
plt.title('Risultato del Particle Filter sulla Localizzazione 2D', fontsize=14)
plt.xlabel('Posizione X [m]', fontsize=12)
plt.ylabel('Posizione Y [m]', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6)
plt.gca().set_aspect('equal', adjustable='box')
plt.show()

---
## Perché all'inizio il Particle Filter è meno preciso? (Fenomeno della Convergenza)

Se osservi attentamente i primissimi passi vicino all'origine $(0, 0)$:
* Le particelle sono state generate casualmente su **tutta** l'area da $0$ a $10$.
* All'istante $t=0$ il filtro non sa ancora dove si trova il robot!
* Servono alcuni passi temporali (e diverse misure consecutive del sensore) affinché le particelle lontane vengano eliminate e quelle rimaste si concentrino densamente attorno al Duckiebot.
* Una volta avvenuta la **convergenza**, la linea verde segue fedelmente e dolcemente la traiettoria reale, filtrando brillantemente l'enorme dispersione dei punti rossi!

Verifichiamolo calcolando l'errore istantaneo di localizzazione:

In [ ]:
err_sensor = np.linalg.norm(measurements - ground_truth[1:], axis=1)
err_pf = np.linalg.norm(filtered_states - ground_truth[1:], axis=1)

plt.figure(figsize=(10, 4.5))
plt.plot(range(1, num_data + 1), err_sensor, 'r--', alpha=0.6, label=f'Errore Sensore Grezzo (Medio: {np.mean(err_sensor):.2f} m)')
plt.plot(range(1, num_data + 1), err_pf, 'g-', linewidth=2, label=f'Errore Particle Filter (Medio: {np.mean(err_pf):.2f} m)')
plt.title('Errore di Localizzazione nel Tempo: Sensore vs Particle Filter', fontsize=13)
plt.xlabel('Passo Temporale (t)', fontsize=11)
plt.ylabel('Errore Euclideo [m]', fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(fontsize=11)
plt.show()

---
# Come "Pensano" le Particelle? Visualizzazione dell'Evoluzione nel Tempo

Per comprendere visivamente il meccanismo interno del Particle Filter, plottiamo 4 istanti temporali salienti ($t=1, t=10, t=50, t=100$):
* Pallino Blu: **Posizione Reale** del Duckiebot.
* Punti Rossi: **Particelle** (la cui dimensione è proporzionale al peso $w$).
* Pallino Verde: **Stima del Filtro**.
* Pallino Arancione: **Misura istantanea rumorosa**.

In [ ]:
snapshot_timesteps = [1, 10, 50, 100]
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for idx, step in enumerate(snapshot_timesteps):
    ax = axes[idx]
    t_idx = step - 1
    true_pt = ground_truth[step]
    filt_pt = filtered_states[t_idx]
    meas_pt = measurements[t_idx]
    ps = particles_over_time[t_idx]
    
    # Particelle rosse con dimensione legata al peso normalizzato
    max_w = max(p.weight for p in ps) if max(p.weight for p in ps) > 0 else 1.0
    for p in ps:
        norm_size = max(5.0, (p.weight / max_w) * 60.0)
        ax.scatter(p.state[0], p.state[1], color='red', s=norm_size, alpha=0.5)
        
    # True, Filtered, Measured
    ax.scatter(true_pt[0], true_pt[1], color='blue', s=100, label='Vero (True)', zorder=5)
    ax.scatter(filt_pt[0], filt_pt[1], color='green', s=100, marker='^', label='Filtro PF', zorder=6)
    ax.scatter(meas_pt[0], meas_pt[1], color='orange', s=80, marker='x', label='Sensore', zorder=7)
    
    ax.set_title(f"Passo t = {step}", fontsize=13)
    ax.set_xlim(-1, 12)
    ax.set_ylim(-1, 12)
    ax.set_aspect('equal')
    ax.grid(True, linestyle=':', alpha=0.5)
    if idx == 0:
        ax.legend(loc='upper left', fontsize=9)

plt.suptitle('Evoluzione della Nuvola di Particelle nel Tempo', fontsize=15)
plt.tight_layout()
plt.show()

---
# Tabella di Confronto: Kalman Filter vs Particle Filter

| Caratteristica | Filtro di Kalman (KF) | Filtro a Particelle (PF) |
| :--- | :--- | :--- |
| **Rappresentazione dello Stato** | Media $\mu$ e Matrice di Covarianza $\Sigma$ | Insieme di $M$ particelle ponderate $\{ (\mathbf{x}^i, w^i) \}$ |
| **Forma della Distribuzione** | Solo Gaussiana unimodale (una campana) | Qualsiasi forma (anche multimodale o irregolare!) |
| **Ipotesi sui Sistemi** | Richiede equazioni lineari e rumore gaussiano | Funziona con qualsiasi fisica e sensore non lineare |
| **Costo Computazionale** | Leggerissimo (poche operazioni su matrici) | Più pesante (dipende dal numero di particelle $M$) |
| **Comportamento Iniziale** | Richiede un belief iniziale ben calibrato | Può partire da zero con particelle uniformi |
| **Ideale per...** | Robot su traiettorie note, tracciamento continuo | Localizzazione globale (*Kidnapped robot*, stanze simmetriche) |

---

### Prossimo Passo: I Filtri su Griglia (Histogram Filter)
Ora che conosci il Filtro di Kalman e il Filtro a Particelle, sei pronto per scoprire come il Duckiebot segue le linee della corsia stradale tramite una griglia discreta!
Procedi pure al terzo notebook: **[Histogram Filter (Filtro a Istogramma)](../03-histogram-filter/histogram_filter.ipynb)**.